# 04l | Drills 2: binary search, sorting and heaps, recursion and linked lists

Covers **04e–04g**: binary search (finding a value, finding a boundary, binary search on the answer), custom sorting and intervals, heaps, recursion, linked lists.

**How to use this lesson:** there are no input boxes here; each problem is followed directly by its answer and output. After reading a problem, think first (in your head or on paper): **which pattern is this?** Then read on. After each problem, cover the answer and try to state the approach and its complexity.

Problems on the same page run in order, so functions defined earlier can be reused later.

The following code has already been run at the top of this page (same as in 04g; in an interview the problem gives you the node definition, so you don't write it yourself):

```python
class ListNode:
    def __init__(self, val, next=None):
        self.val = val
        self.next = next

def build_list(values):
    head = None
    for v in reversed(values):
        head = ListNode(v, head)
    return head

def to_list(head):
    out = []
    cur = head
    while cur is not None:
        out.append(cur.val)
        cur = cur.next
    return out
```

### Problem 1 · Minimum of a rotated array

An ascending array has been "rotated" at some point, e.g. `[4, 5, 6, 7, 0, 1, 2]` (all elements distinct). Find the minimum in O(log n).

```python
def find_min(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1               # the minimum is to the right of mid
        else:
            hi = mid                   # mid itself may be the minimum
    return nums[lo]

print(find_min([4, 5, 6, 7, 0, 1, 2]), find_min([3, 1, 2]), find_min([1, 2, 3]))
```

**Result:**

```text
0 1 1
```

> Compare with the rightmost number: `nums[mid] > nums[hi]` means mid is in the higher left segment, so the minimum must be on the right. O(log n).

### Problem 2 · Search in a rotated array

Find the index of `target` in a rotated array like the one above; return `-1` if it's absent.

```python
def search_rotated(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:              # the left half is sorted
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                  # the right half is sorted
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1

print(search_rotated([4, 5, 6, 7, 0, 1, 2], 0))
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 3))
print(search_rotated([1], 1))
```

**Result:**

```text
4
-1
0
```

> Split at mid: **at least one of the two halves is sorted**. Check whether target falls within the sorted half's range to decide which way to go. `a <= x < b` is Python's chained comparison. O(log n).

### Problem 3 · Find a peak

Return the index of any element "larger than both its neighbors" (positions beyond the ends count as negative infinity). `[1, 2, 1, 3, 5, 6, 4]` → `5` (or `1`).

```python
def find_peak(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1               # walk uphill; there must be a peak ahead
        else:
            hi = mid
    return lo

print(find_peak([1, 2, 3, 1]), find_peak([1, 2, 1, 3, 5, 6, 4]))
```

**Result:**

```text
2 5
```

> Binary search works even on an unsorted array, as long as you can tell "which half the answer is in". Walking uphill always leads to a peak. O(log n).

### Problem 4 · Binary search on the answer: eating bananas

`piles` holds the number of bananas in each pile. Each hour you can eat at most `k` bananas from a single pile. What is the smallest `k` that lets you finish within `h` hours?

`[3, 6, 7, 11], h=8` → `4`.

```python
def min_eating_speed(piles, h):
    def hours(k):
        total = 0
        for p in piles:
            total += (p + k - 1) // k  # round up
        return total

    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if hours(mid) <= h:
            hi = mid                   # mid is fast enough; try slower
        else:
            lo = mid + 1
    return lo

print(min_eating_speed([3, 6, 7, 11], 8), min_eating_speed([30, 11, 23, 4, 20], 5))
```

**Result:**

```text
4 30
```

> The faster you eat, the less time it takes, which is monotonic, so binary search on "speed" for the first value that works. `(p + k - 1) // k` is integer ceiling division. O(n log max).

### Problem 5 · Binary search on the answer: shipping packages

`weights` must be shipped in order within `days` days, and each day's total weight can't exceed the ship's capacity. What is the minimum capacity?

`[1, 2, ..., 10], days=5` → `15`.

```python
def ship_within_days(weights, days):
    def days_needed(cap):
        d = 1
        load = 0
        for w in weights:
            if load + w > cap:
                d += 1
                load = 0
            load += w
        return d

    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if days_needed(mid) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(ship_within_days(list(range(1, 11)), 5), ship_within_days([3, 2, 2, 4, 1, 4], 3))
```

**Result:**

```text
15 6
```

> Same template as the bananas: the lower bound is the heaviest package, the upper bound is shipping everything in one day. The check function is a single greedy pass. O(n log(total weight)).

### Problem 6 · Search a 2D matrix

Each row of the matrix is ascending, and each row's first number is larger than the previous row's last number. Determine whether `target` is present.

```python
def search_matrix(matrix, target):
    rows, cols = len(matrix), len(matrix[0])
    lo, hi = 0, rows * cols - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        x = matrix[mid // cols][mid % cols]    # convert the 1D index to (row, col)
        if x == target:
            return True
        if x < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

m = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
print(search_matrix(m, 3), search_matrix(m, 13))
```

**Result:**

```text
True False
```

> Treat the whole matrix as one long sorted list: element `mid` is at row `mid // cols`, column `mid % cols`. O(log(mn)).

### Problem 7 · Insert interval

`intervals` is sorted by start and non-overlapping. Insert `new` and merge any overlaps.

Inserting `[2, 5]` into `[[1, 3], [6, 9]]` → `[[1, 5], [6, 9]]`.

```python
def insert_interval(intervals, new):
    out = []
    start, end = new
    i = 0
    n = len(intervals)
    while i < n and intervals[i][1] < start:       # entirely to the left
        out.append(intervals[i])
        i += 1
    while i < n and intervals[i][0] <= end:        # overlaps new: merge it in
        start = min(start, intervals[i][0])
        end = max(end, intervals[i][1])
        i += 1
    out.append([start, end])
    out.extend(intervals[i:])                      # entirely to the right
    return out

print(insert_interval([[1, 3], [6, 9]], [2, 5]))
print(insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]))
```

**Result:**

```text
[[1, 5], [6, 9]]
[[1, 2], [3, 10], [12, 16]]
```

> Three parts: non-overlapping ones on the left, the ones overlapping the new interval (merged into one), and non-overlapping ones on the right. O(n).

### Problem 8 · Minimum intervals to remove

What is the minimum number of intervals to remove so that the rest don't overlap? (Touching endpoints don't count as overlap.)

`[[1, 2], [2, 3], [3, 4], [1, 3]]` → `1`.

```python
def erase_overlap(intervals):
    intervals = sorted(intervals, key=lambda x: x[1])   # sort by end time
    keep = 0
    end = float("-inf")
    for s, e in intervals:
        if s >= end:                   # doesn't overlap the last one kept
            keep += 1
            end = e
    return len(intervals) - keep

print(erase_overlap([[1, 2], [2, 3], [3, 4], [1, 3]]), erase_overlap([[1, 2], [1, 2], [1, 2]]))
```

**Result:**

```text
1 2
```

> Greedy: sort by **end time** and always keep the one that ends earliest, which leaves the most room for the rest. Number removed = total − maximum number you can keep. O(n log n).

### Problem 9 · K closest points to the origin

`points = [[1, 3], [-2, 2], [5, 8], [0, 1]]`; return the 2 points closest to the origin.

```python
def k_closest(points, k):
    return sorted(points, key=lambda p: p[0] ** 2 + p[1] ** 2)[:k]

k_closest([[1, 3], [-2, 2], [5, 8], [0, 1]], 2)
```

**Result:**

```text
[[0, 1], [-2, 2]]
```

> No square root needed to compare distances; comparing sums of squares is enough. Sorting is O(n log n); when the data is large and k is small, a size-k heap gives O(n log k).

### Problem 10 · Merge k sorted lists

Merge `[[1, 4, 5], [1, 3, 4], [2, 6]]` into a single ascending list.

```python
import heapq

def merge_k(lists):
    h = []
    for i, xs in enumerate(lists):
        if xs:
            heapq.heappush(h, (xs[0], i, 0))   # (value, which list, index within it)
    out = []
    while h:
        val, i, j = heapq.heappop(h)
        out.append(val)
        if j + 1 < len(lists[i]):
            heapq.heappush(h, (lists[i][j + 1], i, j + 1))
    return out

merge_k([[1, 4, 5], [1, 3, 4], [2, 6]])
```

**Result:**

```text
[1, 1, 2, 3, 4, 4, 5, 6]
```

> The heap holds the current smallest element of each list. Tuples compare by their first element, then the second on ties, so the heap is ordered by value. N numbers in k lists: O(N log k).

### Problem 11 · Last stone weight

Each turn, smash the two heaviest stones together: if they weigh the same, both are destroyed; otherwise the difference remains. Return the weight of the last stone (0 if none remain).

`[2, 7, 4, 1, 8, 1]` → `1`.

```python
import heapq

def last_stone(stones):
    h = []
    for s in stones:
        heapq.heappush(h, -s)          # negating turns the min-heap into a max-heap
    while len(h) > 1:
        a = -heapq.heappop(h)
        b = -heapq.heappop(h)
        if a != b:
            heapq.heappush(h, -(a - b))
    return -h[0] if h else 0

print(last_stone([2, 7, 4, 1, 8, 1]), last_stone([3, 3]))
```

**Result:**

```text
1 0
```

> `heapq` only provides a min-heap. To grab the largest at any time, push the negated values. O(n log n).

### Problem 12 · Sort characters by frequency

Rearrange a string by character frequency, most frequent first; ties go in alphabetical order. `"tree"` → `"eert"`.

```python
from collections import Counter

def frequency_sort(s):
    counts = Counter(s)
    order = sorted(counts, key=lambda ch: (-counts[ch], ch))
    return "".join([ch * counts[ch] for ch in order])

print(frequency_sort("tree"), frequency_sort("cccaaa"), frequency_sort("Aabb"))
```

**Result:**

```text
eert aaaccc bbAa
```

> Same as "top k frequent words" in 04f: negate the count for descending order. `ch * 3` repeats a character 3 times. O(n log n).

### Problem 13 · H-index

`citations` holds each paper's citation count. The H-index is the largest h such that h papers each have at least h citations.

`[3, 0, 6, 1, 5]` → `3`.

```python
def h_index(citations):
    c = sorted(citations, reverse=True)
    h = 0
    for i, x in enumerate(c):
        if x >= i + 1:                 # the (i+1)-th paper (high to low) has at least i+1 citations
            h = i + 1
        else:
            break
    return h

print(h_index([3, 0, 6, 1, 5]), h_index([1, 3, 1]), h_index([0]))
```

**Result:**

```text
3 1 0
```

> After sorting in descending order, if the (i+1)-th paper has ≥ i+1 citations, h is at least i+1. O(n log n).

### Problem 14 · Remove linked list elements

Remove every node whose value equals `val`. `1→2→6→3→6`, remove 6 → `1→2→3`.

```python
def remove_elements(head, val):
    dummy = ListNode(0, head)          # dummy head: no special case when the head must go too
    cur = dummy
    while cur.next is not None:
        if cur.next.val == val:
            cur.next = cur.next.next   # skip the next node
        else:
            cur = cur.next
    return dummy.next

print(to_list(remove_elements(build_list([1, 2, 6, 3, 6]), 6)))
print(to_list(remove_elements(build_list([7, 7, 7]), 7)))
```

**Result:**

```text
[1, 2, 3]
[]
```

> The **dummy head** is the most common linked-list trick: add a node at the front, and deleting the head becomes the same case as deleting a middle node. O(n).

### Problem 15 · Linked list cycle

Determine whether a linked list has a cycle (some node's `next` points back to an earlier node).

```python
def has_cycle(head):
    slow = head
    fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

a = build_list([3, 2, 0, -4])
print(has_cycle(a))
a.next.next.next.next = a.next         # point -4 back at 2 to create a cycle
print(has_cycle(a))
```

**Result:**

```text
False
True
```

> Fast and slow pointers: the fast one moves two steps, the slow one moves one. With a cycle, fast is guaranteed to catch slow; without one, fast reaches `None` first. O(n) time, O(1) space. Use `is` to test whether two references are the same node.

### Problem 16 · Remove the n-th node from the end

`1→2→3→4→5`, remove the 2nd from the end → `1→2→3→5`. Only one pass.

```python
def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    fast = dummy
    slow = dummy
    for _ in range(n):                 # move fast n steps ahead first
        fast = fast.next
    while fast.next is not None:       # then move both; stop when fast is at the last node
        fast = fast.next
        slow = slow.next
    slow.next = slow.next.next         # slow's next node is the one to remove
    return dummy.next

print(to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)))
print(to_list(remove_nth_from_end(build_list([1]), 1)))
```

**Result:**

```text
[1, 2, 3, 5]
[]
```

> Keep the two pointers n steps apart; when the front one reaches the end, the back one stops right before the node to remove. With a dummy head, removing the head needs no special case either. O(n).

### Problem 17 · Palindrome linked list

Determine whether a linked list reads the same forward and backward.

```python
def is_palindrome_list(head):
    vals = to_list(head)
    left, right = 0, len(vals) - 1
    while left < right:
        if vals[left] != vals[right]:
            return False
        left += 1
        right -= 1
    return True

print(is_palindrome_list(build_list([1, 2, 2, 1])), is_palindrome_list(build_list([1, 2])))
```

**Result:**

```text
True False
```

> Read the values into a list, then use opposite-direction two pointers. O(n) time, O(n) space. If asked for O(1) space: find the middle with fast and slow pointers → reverse the second half (04g) → compare the two halves element by element.

### Problem 18 · Permutations

Return all permutations of `[1, 2, 3]`.

```python
def permutations(xs):
    if len(xs) <= 1:
        return [xs[:]]
    out = []
    for i in range(len(xs)):
        rest = xs[:i] + xs[i + 1:]     # remove the i-th element
        for p in permutations(rest):
            out.append([xs[i]] + p)
    return out

permutations([1, 2, 3])
```

**Result:**

```text
[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
```

> Recursion: choose what goes first, then let the function arrange the rest. n numbers have n! permutations, so time is O(n · n!).

### Problem 19 · Subsets

Return all subsets of `[1, 2, 3]`.

```python
def subsets(xs):
    if not xs:
        return [[]]
    rest = subsets(xs[1:])             # all subsets without the first element
    return rest + [[xs[0]] + s for s in rest]

subsets([1, 2, 3])
```

**Result:**

```text
[[], [3], [2], [2, 3], [1], [1, 3], [1, 2], [1, 2, 3]]
```

> Every subset either excludes the first element or includes it. The latter are just the former with the first element added. 2ⁿ subsets, O(n · 2ⁿ).

### Problem 20 · Fast exponentiation

Without `**`, compute `x` to the power `n` (n may be negative) in O(log n).

```python
def my_pow(x, n):
    if n < 0:
        return 1 / my_pow(x, -n)
    if n == 0:
        return 1
    half = my_pow(x, n // 2)
    if n % 2 == 0:
        return half * half
    return half * half * x

print(my_pow(2, 10), my_pow(2, -2), my_pow(3, 0))
```

**Result:**

```text
1024 0.25 1
```

> xⁿ = (x^(n/2))², with one extra factor of x when n is odd. n halves each time, so there are only log n levels of recursion. Note that `half` is computed once; writing `my_pow(x, n // 2) * my_pow(x, n // 2)` degrades to O(n).

## Pattern recap

| Pattern | When to think of it | Key code |
|---|---|---|
| Boundary binary search | First position satisfying a condition | `while lo < hi`; if satisfied `hi = mid`, else `lo = mid + 1` |
| Binary search on partially sorted data | Rotated arrays, peaks | First decide which half is sorted / walk uphill |
| Binary search on the answer | "Minimum speed / capacity" | Write a check function; larger answers are easier to satisfy |
| Intervals | Merge, insert, minimum removals | Sort first; sort by end time for greedy |
| Custom sorting | Multiple sort keys | `sorted(xs, key=lambda x: (-count, x))` |
| Heap | Grab the min / max at any time, k-way merge | `heappush` / `heappop`; store negatives for a max-heap; tuples compare by first element |
| Recursion | Permutations, subsets, halving | Write the base case first; trust it works for smaller inputs |
| Linked list | Removal, n-th from the end, cycle detection | Dummy head, fast and slow pointers |